# When Something Changes

Real-world time series rarely behave nicely. Sensors drift, markets crash, processes
degrade. Detecting **when** something changes is often more valuable than predicting
what comes next.

This notebook walks through the changepoint and anomaly detection toolkit in
`polars_ts`, progressing from classical statistical tests to Bayesian methods and
machine-learning approaches:

| Method | Family | What it finds |
|--------|--------|---------------|
| CUSUM | Classical | Cumulative deviation from a target mean |
| PELT | Optimisation | Optimal set of changepoints (penalised likelihood) |
| BOCPD | Bayesian | Online changepoint probabilities |
| Regime detection | HMM | Latent state switches |
| Z-score / IQR | Statistical | Point anomalies |
| Isolation Forest | ML | Multivariate anomalies |
| Fourier decomposition | Spectral | Anomalies in the residual after detrending |

We use two synthetic series with injected mean shifts, trend breaks, and point
anomalies so that every method has something interesting to find.

In [ ]:
# Install polars-timeseries if not already available
import importlib

if importlib.util.find_spec("polars_ts") is None:
    %pip install -q polars-timeseries[all]

In [ ]:
try:
    import hvplot.polars  # noqa
except ImportError:
    pass  # hvplot is optional for visualization
import numpy as np
import polars as pl

from polars_ts import (
    bocpd,
    cusum,
    detect_outliers,
    fourier_decomposition,
    isolation_forest_detect,
    pelt,
    regime_detect,
    treat_outliers,
)

## Synthetic data

**Series S1** has two abrupt mean shifts (at *t = 200* and *t = 350*) plus three
injected point anomalies.

**Series S2** features a gradual upward trend that reverses at *t = 250*, combined
with a volatility regime change (low noise before the break, high noise after).

In [ ]:
np.random.seed(42)
n = 500

# Series 1: mean shifts at t=200 and t=350, plus point anomalies
y1 = np.concatenate(
    [
        np.random.normal(10, 1, 200),
        np.random.normal(15, 1.5, 150),
        np.random.normal(8, 1, 150),
    ]
)
y1[50] = 25
y1[100] = -5
y1[300] = 30

# Series 2: trend reversal at t=250 with volatility regime change
t = np.arange(n)
y2 = np.where(
    t < 250,
    0.02 * t + np.random.normal(0, 0.5, n),
    -0.03 * (t - 250) + 5 + np.random.normal(0, 2.0, n),
)

df = pl.DataFrame(
    {
        "unique_id": ["S1"] * n + ["S2"] * n,
        "ds": list(range(n)) * 2,
        "y": np.concatenate([y1, y2]),
    }
)

df.head()

In [ ]:
# Quick look at the raw series
df.hvplot.line(x="ds", y="y", by="unique_id", width=800, height=350, title="Raw synthetic series")

## 1. CUSUM (Cumulative Sum)

The CUSUM statistic accumulates deviations from the series mean. A sustained
change in level causes the cumulative sum to trend away from zero, making
structural breaks visually obvious.

Reference: E. S. Page, *Continuous Inspection Schemes*, Biometrika, 1954.

In [ ]:
df_cusum = cusum(df, normalize=True)
df_cusum.head()

In [ ]:
df_cusum.hvplot.line(
    x="ds",
    y="cusum",
    by="unique_id",
    width=800,
    height=350,
    title="CUSUM statistic (normalised)",
    ylabel="Cumulative sum",
)

## 2. PELT (Pruned Exact Linear Time)

PELT finds the optimal set of changepoints by minimising a penalised cost
function. It is *exact* (not approximate) yet runs in expected O(n) time.

Reference: R. Killick, P. Fearnhead & I. A. Eckley, *Optimal Detection of
Changepoints with a Linear Computational Cost*, JASA, 2012.

In [ ]:
df_pelt = pelt(df, cost="mean", penalty=None, min_size=2)
df_pelt

In [ ]:
# Overlay changepoints on the raw series for S1
s1 = df.filter(pl.col("unique_id") == "S1")
cp_s1 = df_pelt.filter(pl.col("unique_id") == "S1")["ds"].to_list()

line = s1.hvplot.line(x="ds", y="y", width=800, height=350, title="PELT changepoints (S1)")
vlines = line * s1.filter(pl.col("ds").is_in(cp_s1)).hvplot.scatter(
    x="ds", y="y", color="red", size=80, marker="triangle", label="changepoint"
)
vlines

## 3. BOCPD (Bayesian Online Changepoint Detection)

BOCPD maintains a posterior distribution over *run lengths* (how long since
the last changepoint). When the run-length probability mass collapses back to
zero, a changepoint has been detected.

Reference: R. P. Adams & D. J. C. MacKay, *Bayesian Online Changepoint
Detection*, arXiv:0710.3742, 2007.

In [ ]:
df_bocpd = bocpd(df, hazard_rate=200.0, threshold=0.5)
df_bocpd.filter(pl.col("is_changepoint")).head(10)

In [ ]:
# Changepoint probability over time for S1
bocpd_s1 = df_bocpd.filter(pl.col("unique_id") == "S1")
bocpd_s1.hvplot.line(
    x="ds",
    y="changepoint_prob",
    width=800,
    height=300,
    title="BOCPD changepoint probability (S1)",
    ylabel="P(changepoint)",
)

## 4. Regime detection (Hidden Markov Model)

A Gaussian HMM is fitted to identify latent regimes. Each observation is
assigned to the most probable state, and transitions between states mark
regime changes.

We ask for three states to capture the three mean levels in S1 and the
trend/volatility regimes in S2.

In [ ]:
df_regime = regime_detect(df, n_states=3, seed=42)
df_regime.head()

In [ ]:
# Colour the series by detected regime
regime_s1 = df_regime.filter(pl.col("unique_id") == "S1")
regime_s1.hvplot.scatter(
    x="ds",
    y="y",
    color="regime",
    width=800,
    height=350,
    cmap="Category10",
    title="HMM regime detection (S1, 3 states)",
)

## 5. Outlier detection: Z-score and IQR

Statistical outlier detectors flag individual observations that deviate
significantly from the bulk of the data.

- **Z-score**: flags points more than *k* standard deviations from the mean.
- **IQR**: flags points outside the interquartile fence (Q1 - 1.5*IQR, Q3 + 1.5*IQR).

In [ ]:
df_zscore = detect_outliers(df, method="zscore", threshold=3.0)
df_iqr = detect_outliers(df, method="iqr")

print("Z-score outliers:", df_zscore.filter(pl.col("is_outlier")).shape[0])
print("IQR outliers:   ", df_iqr.filter(pl.col("is_outlier")).shape[0])

In [ ]:
# Visualise Z-score outliers for S1
zs_s1 = df_zscore.filter(pl.col("unique_id") == "S1")
zs_s1.hvplot.scatter(
    x="ds",
    y="y",
    color="is_outlier",
    width=800,
    height=350,
    cmap={True: "red", False: "steelblue"},
    title="Z-score outlier detection (S1, threshold=3.0)",
)

## 6. Outlier treatment

Once outliers are identified, we need a strategy for replacing them. Two
common approaches:

- **Clip**: cap values at the detection boundary (preserves order).
- **Interpolate**: replace outliers with linearly interpolated values from
  their neighbours (preserves trend continuity).

In [ ]:
df_clipped = treat_outliers(df, method="zscore", replacement="clip")
df_interp = treat_outliers(df, method="zscore", replacement="interpolate")

# Compare on S1
clip_s1 = df_clipped.filter(pl.col("unique_id") == "S1")
interp_s1 = df_interp.filter(pl.col("unique_id") == "S1")

(
    s1.hvplot.line(x="ds", y="y", label="original", width=800, height=350, alpha=0.4)
    * clip_s1.hvplot.line(x="ds", y="y", label="clipped", color="orange")
    * interp_s1.hvplot.line(x="ds", y="y", label="interpolated", color="green")
).opts(title="Outlier treatment comparison (S1)")

## 7. Isolation Forest

Isolation Forest detects anomalies by measuring how easily a point can be
isolated via random axis-aligned splits. Anomalous points require fewer
splits and therefore have higher anomaly scores.

Reference: F. T. Liu, K. M. Ting & Z.-H. Zhou, *Isolation Forest*, ICDM, 2008.

In [ ]:
df_iso = isolation_forest_detect(df, feature_cols=["y"], contamination=0.05)
df_iso.filter(pl.col("is_anomaly")).head(10)

In [ ]:
# Anomaly score heatmap for S1
iso_s1 = df_iso.filter(pl.col("unique_id") == "S1")
iso_s1.hvplot.scatter(
    x="ds",
    y="y",
    color="anomaly_score",
    width=800,
    height=350,
    cmap="RdYlGn_r",
    title="Isolation Forest anomaly scores (S1)",
    colorbar=True,
)

## 8. Fourier decomposition-based anomaly detection

Fourier decomposition separates the signal into trend and seasonal components.
The residual (observed minus reconstructed) is then tested for anomalies:
points where the residual exceeds `anomaly_threshold` standard deviations
are flagged.

This is particularly effective for series with strong periodicity where
point anomalies hide inside seasonal swings.

In [ ]:
df_fourier = fourier_decomposition(df, ts_freq=365, anomaly_threshold=2.0)
df_fourier.head()

In [ ]:
# Highlight Fourier-detected anomalies on S1
fourier_s1 = df_fourier.filter(pl.col("unique_id") == "S1")
anomalies_s1 = fourier_s1.filter(pl.col("is_anomaly"))

(
    fourier_s1.hvplot.line(x="ds", y="y", width=800, height=350, label="series")
    * anomalies_s1.hvplot.scatter(x="ds", y="y", color="red", size=60, label="anomaly")
).opts(title="Fourier decomposition anomaly detection (S1)")

## Summary

We covered seven complementary approaches to detecting structural changes and
anomalies in time series:

1. **CUSUM** gives a quick visual diagnostic for sustained mean shifts.
2. **PELT** provides the statistically optimal set of changepoints.
3. **BOCPD** offers a probabilistic, online-friendly alternative.
4. **Regime detection** via HMM captures latent state dynamics.
5. **Z-score / IQR** detectors handle simple point anomalies.
6. **Isolation Forest** extends anomaly detection to multivariate settings.
7. **Fourier decomposition** catches anomalies hiding in seasonal patterns.

In practice, combining multiple methods (e.g., PELT for changepoints plus
Isolation Forest for point anomalies) gives the most robust results.

### References

- Page, E. S. (1954). *Continuous Inspection Schemes.* Biometrika, 41(1/2), 100-115.
- Killick, R., Fearnhead, P., & Eckley, I. A. (2012). *Optimal Detection of Changepoints with a Linear Computational Cost.* JASA, 107(500), 1590-1598.
- Adams, R. P., & MacKay, D. J. C. (2007). *Bayesian Online Changepoint Detection.* arXiv:0710.3742.
- Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). *Isolation Forest.* Proc. ICDM, 413-422.
- Jain, A. & Lemos, S. (2024). *Modern Time Series Forecasting with Python*, 2nd Ed. Packt.